# Notebook 02 — Create Late-Delivery Labels

This notebook reads the one-row-per-order ML table created by Notebook 01 and creates the binary late-delivery target.

Goals:
- Read and validate the joined ML table
- Identify orders with valid delivery ground truth
- Compare actual delivery date with estimated delivery date
- Create `delay_days`
- Create the binary target `is_late`
- Validate the label on real orders
- Measure class distribution and imbalance
- Save the labeled table for Notebook 03

Target:
- `0` = On time or early
- `1` = Late

Artifact:
`artifacts/02_labeled/labeled_table.parquet`

In [1]:
from pathlib import Path

import pandas as pd


pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    220
)


def find_project_root():

    current_path = Path.cwd().resolve()

    candidate_roots = [
        current_path,
        *current_path.parents,
    ]

    for candidate in candidate_roots:

        if (
            (candidate / "compose.yaml").exists()
            and
            (candidate / "requirements.txt").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Olist-MLOps project root. "
        "Run Jupyter from inside the project directory."
    )


PROJECT_ROOT = find_project_root()


INPUT_PATH = (
    PROJECT_ROOT
    / "artifacts"
    / "01_joined"
    / "ml_table.parquet"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "02_labeled"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


assert INPUT_PATH.exists(), (
    f"Notebook 01 artifact not found: {INPUT_PATH}"
)


print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "Input       :",
    INPUT_PATH
)

print(
    "Input exists:",
    INPUT_PATH.exists()
)

print(
    "Output      :",
    OUTPUT_DIR
)

Project root: G:\(01)04\Qafza_MLOps\Olist-MLOps
Input       : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\01_joined\ml_table.parquet
Input exists: True
Output      : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\02_labeled


In [2]:
df = pd.read_parquet(
    INPUT_PATH
)

print(
    "Joined ML table loaded successfully."
)

print(
    "Shape:",
    df.shape
)

Joined ML table loaded successfully.
Shape: (99441, 44)


In [3]:
print(
    "Rows:",
    len(df)
)

print(
    "Columns:",
    df.shape[1]
)

print(
    "Unique order_id:",
    df["order_id"].nunique()
)

print(
    "Duplicate order_id:",
    df["order_id"]
    .duplicated()
    .sum()
)

assert (
    df["order_id"]
    .duplicated()
    .sum()
    == 0
), "Duplicate order_id detected."

assert (
    df["order_id"]
    .nunique()
    == len(df)
), "Input is not one row per order."

print(
    "\nInput artifact validation passed."
)

Rows: 99441
Columns: 44
Unique order_id: 99441
Duplicate order_id: 0

Input artifact validation passed.


In [4]:
label_columns = [
    "order_id",
    "order_status",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

missing_label_columns = [
    column
    for column in label_columns
    if column not in df.columns
]

assert not missing_label_columns, (
    f"Missing required label columns: "
    f"{missing_label_columns}"
)

display(
    df[
        label_columns
    ].head()
)

label_missing_summary = (
    df[label_columns]
    .isna()
    .sum()
)

print(
    "\nMissing values:"
)

display(
    label_missing_summary
)

,order_id,order_status,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-16 18:17:02,2018-02-26 00:00:00



Missing values:


order_id                            0
order_status                        0
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [5]:
date_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for column in date_columns:

    df[column] = pd.to_datetime(
        df[column],
        errors="coerce"
    )

print(
    df[
        date_columns
    ].dtypes
)

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [6]:
labelable_mask = (
    (df["order_status"] == "delivered")
    & df[
        "order_delivered_customer_date"
    ].notna()
    & df[
        "order_estimated_delivery_date"
    ].notna()
)

labeled_df = (
    df[
        labelable_mask
    ]
    .copy()
)

original_orders = len(df)
labelable_orders = len(labeled_df)

removed_orders = (
    original_orders
    - labelable_orders
)

print(
    "Original orders :",
    f"{original_orders:,}"
)

print(
    "Labelable orders:",
    f"{labelable_orders:,}"
)

print(
    "Removed orders  :",
    f"{removed_orders:,}"
)

Original orders : 99,441
Labelable orders: 96,470
Removed orders  : 2,971


## Label Definition

The target is based on the actual customer delivery date compared with the estimated delivery date.

The comparison is performed at calendar-day level:

- `delay_days < 0` → delivered early
- `delay_days = 0` → delivered on the estimated day
- `delay_days > 0` → delivered late

Binary target:

- `is_late = 0` → on time or early
- `is_late = 1` → late

Only delivered orders with both actual and estimated delivery dates are used because other orders do not have valid delivery ground truth.

In [7]:
actual_date = (
    labeled_df[
        "order_delivered_customer_date"
    ]
    .dt.normalize()
)

estimated_date = (
    labeled_df[
        "order_estimated_delivery_date"
    ]
    .dt.normalize()
)

labeled_df[
    "delay_days"
] = (
    actual_date
    - estimated_date
).dt.days

In [8]:
labeled_df[
    "is_late"
] = (
    labeled_df[
        "delay_days"
    ] > 0
).astype(
    "int8"
)

In [9]:
late_examples = (
    labeled_df.loc[
        labeled_df[
            "is_late"
        ] == 1,
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delay_days",
            "is_late",
        ],
    ]
    .head(5)
)

print(
    "Late-order examples:"
)

display(
    late_examples
)

Late-order examples:


,order_id,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_late
20,203096f03d82e0dffbc41ebc2e2bcfb7,2017-10-09 22:23:46,2017-09-28,11,1
25,fbf9ac61453ac646ce8ad9783d7d0af6,2018-03-21 22:03:54,2018-03-12,9,1
41,6ea2f835b4556291ffdc53fa0b3b95e8,2017-12-28 18:59:23,2017-12-21,7,1
57,66e4624ae69e7dc89bd50222b59f581f,2018-04-03 13:28:46,2018-04-02,1,1
58,a685d016c8a26f71a0bb67821070e398,2017-04-06 13:37:16,2017-03-30,7,1


In [10]:
on_time_examples = (
    labeled_df.loc[
        labeled_df[
            "is_late"
        ] == 0,
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "delay_days",
            "is_late",
        ],
    ]
    .head(5)
)

print(
    "On-time / early examples:"
)

display(
    on_time_examples
)

On-time / early examples:


,order_id,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,-8,0
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,-6,0
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,-18,0
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,-13,0
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,-10,0


In [11]:
invalid_late = (
    labeled_df[
        (
            labeled_df[
                "is_late"
            ] == 1
        )
        & (
            labeled_df[
                "delay_days"
            ] <= 0
        )
    ]
)

invalid_on_time = (
    labeled_df[
        (
            labeled_df[
                "is_late"
            ] == 0
        )
        & (
            labeled_df[
                "delay_days"
            ] > 0
        )
    ]
)

print(
    "Invalid late labels    :",
    len(invalid_late)
)

print(
    "Invalid on-time labels :",
    len(invalid_on_time)
)

assert (
    len(invalid_late) == 0
)

assert (
    len(invalid_on_time) == 0
)

assert (
    labeled_df[
        "delay_days"
    ].isna().sum()
    == 0
)

assert set(
    labeled_df[
        "is_late"
    ].unique()
) == {0, 1}

print(
    "\nLabel validation passed."
)

Invalid late labels    : 0
Invalid on-time labels : 0

Label validation passed.


In [12]:
class_counts = (
    labeled_df[
        "is_late"
    ]
    .value_counts()
    .sort_index()
)

class_percentages = (
    labeled_df[
        "is_late"
    ]
    .value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

class_distribution = pd.DataFrame({
    "count":
        class_counts,

    "percentage":
        class_percentages,
})

class_distribution.index = [
    "On Time",
    "Late",
]

display(
    class_distribution
)

,count,percentage
On Time,89936,93.22691
Late,6534,6.77309


In [13]:
late_count = int(
    (
        labeled_df[
            "is_late"
        ] == 1
    ).sum()
)

on_time_count = int(
    (
        labeled_df[
            "is_late"
        ] == 0
    ).sum()
)

late_percentage = (
    late_count
    / len(labeled_df)
    * 100
)

imbalance_ratio = (
    on_time_count
    / late_count
)

print(
    f"On-time orders : "
    f"{on_time_count:,}"
)

print(
    f"Late orders    : "
    f"{late_count:,}"
)

print(
    f"Late percentage: "
    f"{late_percentage:.2f}%"
)

print(
    f"Imbalance ratio: "
    f"{imbalance_ratio:.2f}:1"
)

On-time orders : 89,936
Late orders    : 6,534
Late percentage: 6.77%
Imbalance ratio: 13.76:1


In [14]:
if imbalance_ratio >= 2:

    print(
        "\nConclusion:"
        " The target has a clear class imbalance."
    )

else:

    print(
        "\nConclusion:"
        " No strong class imbalance detected."
    )


Conclusion: The target has a clear class imbalance.


In [15]:
delay_summary = pd.DataFrame({
    "all_delivered_orders":
        labeled_df[
            "delay_days"
        ].describe(),

    "late_orders_only":
        labeled_df.loc[
            labeled_df[
                "is_late"
            ] == 1,
            "delay_days",
        ].describe(),
})

display(
    delay_summary
)

,all_delivered_orders,late_orders_only
count,96470.000000,6534.000000
mean,-11.875889,10.620141
std,10.182105,14.644955
min,-147.000000,1.000000
25%,-17.000000,3.000000
50%,-12.000000,7.000000
75%,-7.000000,13.000000
max,188.000000,188.000000


In [16]:
print(
    "Rows:",
    len(labeled_df)
)

print(
    "Columns:",
    labeled_df.shape[1]
)

print(
    "Unique order_id:",
    labeled_df[
        "order_id"
    ].nunique()
)

print(
    "Duplicate order_id:",
    labeled_df[
        "order_id"
    ].duplicated().sum()
)

print(
    "Missing labels:",
    labeled_df[
        "is_late"
    ].isna().sum()
)

assert (
    labeled_df[
        "order_id"
    ].nunique()
    == len(labeled_df)
)

assert (
    labeled_df[
        "order_id"
    ].duplicated().sum()
    == 0
)

assert (
    labeled_df[
        "is_late"
    ].isna().sum()
    == 0
)

print(
    "\nLabeled-table validation passed."
)

Rows: 96470
Columns: 46
Unique order_id: 96470
Duplicate order_id: 0
Missing labels: 0

Labeled-table validation passed.


In [17]:
LABELED_PATH = (
    OUTPUT_DIR
    / "labeled_table.parquet"
)

labeled_df.to_parquet(
    LABELED_PATH,
    index=False,
)

print(
    "Saved artifact:"
)

print(
    LABELED_PATH
)

Saved artifact:
G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\02_labeled\labeled_table.parquet


In [18]:
saved_labeled_df = (
    pd.read_parquet(
        LABELED_PATH
    )
)

print(
    "Saved shape:",
    saved_labeled_df.shape
)

print(
    "Unique order_id:",
    saved_labeled_df[
        "order_id"
    ].nunique()
)

print(
    "Duplicate order_id:",
    saved_labeled_df[
        "order_id"
    ].duplicated().sum()
)

print(
    "Missing labels:",
    saved_labeled_df[
        "is_late"
    ].isna().sum()
)

assert (
    saved_labeled_df.shape
    == labeled_df.shape
)

assert (
    saved_labeled_df[
        "order_id"
    ].nunique()
    == len(saved_labeled_df)
)

assert (
    saved_labeled_df[
        "order_id"
    ].duplicated().sum()
    == 0
)

assert (
    saved_labeled_df[
        "is_late"
    ].isna().sum()
    == 0
)

print(
    "\nSaved artifact validation passed."
)

Saved shape: (96470, 46)
Unique order_id: 96470
Duplicate order_id: 0
Missing labels: 0

Saved artifact validation passed.


In [19]:
print(
    "=" * 60
)

print(
    "NOTEBOOK 02 COMPLETE"
)

print(
    "=" * 60
)

print(
    f"Input orders       : "
    f"{len(df):,}"
)

print(
    f"Labelable orders   : "
    f"{len(saved_labeled_df):,}"
)

print(
    f"Removed orders     : "
    f"{len(df) - len(saved_labeled_df):,}"
)

print(
    f"On-time orders     : "
    f"{on_time_count:,}"
)

print(
    f"Late orders        : "
    f"{late_count:,}"
)

print(
    f"Late percentage    : "
    f"{late_percentage:.2f}%"
)

print(
    f"Imbalance ratio    : "
    f"{imbalance_ratio:.2f}:1"
)

print(
    f"Columns            : "
    f"{saved_labeled_df.shape[1]}"
)

print(
    f"Duplicate order_id : "
    f"{saved_labeled_df['order_id'].duplicated().sum()}"
)

print(
    f"Missing labels     : "
    f"{saved_labeled_df['is_late'].isna().sum()}"
)

print(
    f"Artifact           : "
    f"{LABELED_PATH}"
)

NOTEBOOK 02 COMPLETE
Input orders       : 99,441
Labelable orders   : 96,470
Removed orders     : 2,971
On-time orders     : 89,936
Late orders        : 6,534
Late percentage    : 6.77%
Imbalance ratio    : 13.76:1
Columns            : 46
Duplicate order_id : 0
Missing labels     : 0
Artifact           : G:\(01)04\Qafza_MLOps\Olist-MLOps\artifacts\02_labeled\labeled_table.parquet
